Download de bibliotecas e Login no site

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import Select
from datetime import datetime, timedelta
import calendar

#biblioteca para fazer o navegador nao fechar no fim do código
options = Options()
options.add_experimental_option("detach", True)

#URL e login/senha utilizdos para logar no site
personal_url = "URL"
login = ["USUARIO","SENHA"]



In [2]:
                         #defino options=options para o driver receber a opção de não fechar
driver = webdriver.Chrome(options=options)
driver.get(personal_url)

wait = WebDriverWait(driver, 15)  # Espera no máximo 15 segundos

#tentando fazer um hold click com a biblioteca ActionChains #até o momento nao utilizei
acoes = ActionChains(driver)

driver.implicitly_wait(2)

#login no site
bt_login = driver.find_element(By.ID,"login-box-content-user")
bt_login.send_keys(login[0])
bt_senha = driver.find_element(By.ID,"login-box-content-password")
bt_senha.send_keys(login[1])
temp = wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/form/div[5]/div[3]/div[2]/div/div/div/div[2]/div[2]/div/button")))
temp.click()
temp = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Tickets attractions")))
temp.click()

In [3]:
#vetores para os dados que serão coletados no WEB Scraping
nomes_ttk = []
preco_adulto = []
data_ttk = []
preco_adulto_kid = []
preco_kid = []

In [4]:
dataIFormatada = pd.Timestamp.today()
dataFFormatada = pd.Timestamp.today()
dataIFormatada += timedelta(days=1)

Informações ingresso Adulto + KID

In [ ]:
#Dentro da área de busca do site <

dataFFormatada = pd.to_datetime(dataFFormatada)
dataIFormatada = pd.to_datetime(dataIFormatada)

temp = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Tickets attractions")))
temp.click()

#acessando a barra de pesquisa
destino = wait.until(EC.presence_of_element_located((By.ID, "service-searcher-_ctl0_ctlZoneSelector-fake-input")))
destino.click()

destino.send_keys("Walt Disney")

temp = wait.until(EC.presence_of_element_located((By.ID, "service-searcher-_ctl0_ctlZoneSelector-input_listbox")))
temp.click()

#completando a data inicial do calendario
dataInicial = driver.find_element(By.ID, "service-searcher-_ctl0_ctlDateSelector-start-date-input")
dataInicial.click()

dataIFormatada = dataIFormatada.strftime('%m/%d/%Y')
dataInicial.clear()
dataInicial.send_keys(dataIFormatada)

#completando a data final do calendario
dataFinal = driver.find_element(By.ID, "service-searcher-_ctl0_ctlDateSelector-end-date-input")
dataFinal.click()
ult_dia = calendar.monthrange(dataFFormatada.year, dataFFormatada.month)[1]
dataUltDia = datetime(dataFFormatada.year, dataFFormatada.month, ult_dia)
dataFFormatada = dataUltDia.strftime('%m/%d/%Y')
dataFinal.clear()
dataFinal.send_keys(dataFFormatada)


#clica no botao passageiros
driver.find_element(By.ID, "service-searcher-_ctl0_ctlRoomSelector-room-selector-button").click()


lista = wait.until(
    EC.element_to_be_clickable((By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/fielset/ul/li/div/div[1]/div[1]/select"))
)

# Usando o Select para interagir com o dropdown
select = Select(lista)

# Espera até que as opções sejam carregadas, então seleciona a opção 1
# Se a opção for um índice, a opção 0 seria a primeira opção.
wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/fielset/ul/li/div/div[1]/div[1]/select/option[1]")))

# Seleciona a primeira opção
select.select_by_index(0)  # 0 refere-se à primeira opção


lista = wait.until(
    EC.element_to_be_clickable((By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/fielset/ul/li/div/div[1]/div[2]/select"))
)

# Usando o Select para interagir com o dropdown
select = Select(lista)

# Espera até que as opções sejam carregadas, então seleciona a opção 1
# Se a opção for um índice, a opção 0 seria a primeira opção.
wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/fielset/ul/li/div/div[1]/div[1]/select/option[2]")))

# Seleciona a primeira opção
select.select_by_index(1)  # 0 refere-se à primeira opção


#Clica no "Apply"
driver.find_element(By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/footer/button").click()


#Clica em "Buscar"
driver.find_element(By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[2]/button").click()


Carregando todos ingressos da página e clicando em todos os botões de "Exibir mais"

In [38]:
#Carregando a pagina por completo
url = driver.current_url
for page in range(2, 7):
    nova_url = f"{url}#?page={page}"
    driver.get(nova_url)
driver.refresh()

#Clicando em todos os botões de "exibir mais"
botoesExibir = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".btn.show-more-options.result__view-more-button")))

for botao in botoesExibir:
    botao.click()
    time.sleep(1)

In [ ]:
#Coletando os dados

#elementos puxados do site
nomes = driver.find_elements(By.CLASS_NAME, 'result-option__title')
datas = driver.find_elements(By.CLASS_NAME, 'result-option__date')
precos = driver.find_elements(By.CLASS_NAME, 'result-option-price')

#for para os nomes dos tickets
for nome in nomes:
    nomes_ttk.append(nome.text)

#for para as datas dos tickets
for data in datas:
    data = data.text 
    data = data.split()[1]
    data = datetime.strptime(data, '%m/%d/%Y').strftime('%d/%m/%Y')
    data_ttk.append(data)

#for para os preços dos tickets
for preco in precos:
    preco = preco.text
    preco = preco.split()[0]
    preco = preco.replace(",", "")
    preco = float(preco)
    preco_adulto_kid.append(preco)

print(nomes_ttk, data_ttk, preco_adulto_kid)


In [40]:
#algo para voltar na tela inicial
temp = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Home")))
temp.click()


Informações ingressos adulto

In [ ]:
from selenium.webdriver.support.ui import Select

dataFFormatada = pd.to_datetime(dataFFormatada)
dataIFormatada = pd.to_datetime(dataIFormatada)

temp = wait.until(EC.presence_of_element_located((By.LINK_TEXT, "Tickets attractions")))
temp.click()


destino = wait.until(EC.presence_of_element_located((By.ID, "service-searcher-_ctl0_ctlZoneSelector-fake-input")))
destino.click()


destino.send_keys("Walt Disney")

driver.find_element(By.ID, "service-searcher-_ctl0_ctlZoneSelector-input_listbox").click()

#completando a data inicial do calendario
dataInicial = driver.find_element(By.ID, "service-searcher-_ctl0_ctlDateSelector-start-date-input")
dataInicial.click()

dataIFormatada = pd.to_datetime(dataIFormatada)
dataIFormatada = dataIFormatada.strftime('%m/%d/%Y')
dataInicial.clear()
dataInicial.send_keys(dataIFormatada)

#completando a data final do calendario
dataFinal = driver.find_element(By.ID, "service-searcher-_ctl0_ctlDateSelector-end-date-input")
dataFinal.click()
#dataFFormatada = datetime.now()
ult_dia = calendar.monthrange(dataFFormatada.year, dataFFormatada.month)[1]

dataUltDia = datetime(dataFFormatada.year, dataFFormatada.month, ult_dia)
dataFFormatada = dataUltDia.strftime('%m/%d/%Y')
dataFinal.clear()
dataFinal.send_keys(dataFFormatada)


#clica no botao passageiros
driver.find_element(By.ID, "service-searcher-_ctl0_ctlRoomSelector-room-selector-button").click()

# Espera a lista de seleção estar visível e clicável
lista = wait.until(
    EC.element_to_be_clickable((By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/fielset/ul/li/div/div[1]/div[2]/select"))
)

# Usando o Select para interagir com o dropdown
select = Select(lista)

# Espera até que as opções sejam carregadas, então seleciona a opção 1
# Se a opção for um índice, a opção 0 seria a primeira opção.
wait.until(EC.presence_of_element_located((By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/fielset/ul/li/div/div[1]/div[1]/select/option[1]")))

# Seleciona a primeira opção
select.select_by_index(0)  # 0 refere-se à primeira opção                                         

#Clica no "Apply"
driver.find_element(By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[1]/div[1]/div[3]/div/div/div/div/div[2]/div/footer/button").click()

#Clica em "Buscar" via javascript (esse botao simplesmente n vai sem ser pelo js, entao sei la..)
btn_Buscar = driver.find_element(By.XPATH, "/html/body/form/main/div/header/div[1]/section/div[2]/div/div/div[2]/div/div[2]/div[2]/button")
driver.execute_script("arguments[0].click();", btn_Buscar)

In [42]:
#Carregando a pagina por completo
url = driver.current_url
for page in range(2, 7):
    nova_url = f"{url}#?page={page}"
    driver.get(nova_url)
driver.refresh()

#Clicando em todos os botões de "exibir mais"
botoesExibir = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".btn.show-more-options.result__view-more-button")))

for botao in botoesExibir:
    botao.click()
    time.sleep(1)

In [ ]:
#Coletando os dados

#elementos puxados do site
nomes = driver.find_elements(By.CLASS_NAME, 'result-option__title')
datas = driver.find_elements(By.CLASS_NAME, 'result-option__date')
precos = driver.find_elements(By.CLASS_NAME, 'result-option-price')




#for para os preços dos tickets
for preco in precos:
    preco = preco.text
    preco = preco.split()[0]
    preco = preco.replace(",", "")
    preco = float(preco)
    preco_adulto.append(preco)

print(nomes_ttk, data_ttk, preco_adulto)


In [28]:
#Criar algo para icrementar a data
dataFFormatada = pd.to_datetime(dataFFormatada)
dataIFormatada = pd.to_datetime(dataIFormatada)


dataFFormatada += pd.DateOffset(months=1)
dataIFormatada += pd.DateOffset(months=1)
    
dataIFormatada = dataIFormatada.replace(day=1)

print(dataIFormatada, dataFFormatada)

2025-09-01 00:00:00 2025-09-30 00:00:00


In [44]:
preco_kid = [x - y for x, y in zip(preco_adulto_kid, preco_adulto)]
len(preco_kid)

3148

In [45]:
len(nomes_ttk)
len(data_ttk)
len(preco_adulto_kid)
len(preco_adulto)
len(preco_kid)

3148

Criando dataframe com as informações dos ingressos

In [46]:
#Criando o dataframe com os preços, nomes e datas dos tickets
tabela = pd.DataFrame({
    'Nome Tickets': nomes_ttk,
    'Data Ingresso': data_ttk,
    'Preço adulto + criança (USD)': preco_adulto_kid,
    'Preço adulto (USD)': preco_adulto,
    'Preço criança (USD)': preco_kid
})

In [47]:
tabela.to_excel('dados3.xlsx', index=False)